# Data Preprocessing for Loan Risk Prediction

This notebook performs:
- dataset loading
- data cleaning
- missing value handling
- target variable creation
- preprocessing preparation

In [1]:
import pandas as pd
import numpy as np

## Load Dataset

In [2]:
df = pd.read_csv(
    r"D:\Dual Stage Loan Risk Prediction\accepted_2007_to_2018Q4.csv",
    low_memory=False
)

df = df.sample(n=100000, random_state=42)

print("Sampled shape:", df.shape)

Sampled shape: (100000, 151)


## Dataset Overview

In [3]:
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
392949,39651438,NaN,32000.0,32000.0,32000.0,60 months,10.49,687.65,B,B3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1273506,16411620,NaN,9600.0,9600.0,9600.0,36 months,12.99,323.42,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
324024,45122316,NaN,4000.0,4000.0,4000.0,36 months,6.68,122.93,A,A3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2066630,125356772,NaN,6025.0,6025.0,6025.0,36 months,10.91,197.00,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
477199,128490686,NaN,25000.0,25000.0,25000.0,60 months,26.30,752.96,E,E5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


## Remove Identification Features

In [4]:
identification_cols = [
    "id",
    "member_id",
    "url",
    "title"
]

df.drop(columns=identification_cols, inplace=True, errors="ignore")

print("After dropping identification columns:", df.shape)


After dropping identification columns: (100000, 147)


## Remove Irrelevant Features

In [5]:
payment_timeline_cols = [
    "last_pymnt_d",
    "last_pymnt_amnt",
    "next_pymnt_d",
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_interest",
    "total_rec_late_fee"
]

df.drop(columns=payment_timeline_cols, inplace=True, errors="ignore")

print("After dropping payment/timeline columns:", df.shape)


After dropping payment/timeline columns: (100000, 140)


In [6]:
settlement_recovery_cols = [
    "recoveries",
    "collection_recovery_fee",
    "debt_settlement_flag",
    "debt_settlement_flag_date",
    "settlement_status",
    "settlement_date",
    "settlement_amount",
    "settlement_percentage",
    "settlement_term"
]

df.drop(columns=settlement_recovery_cols, inplace=True, errors="ignore")

print("After dropping settlement/recovery columns:", df.shape)


After dropping settlement/recovery columns: (100000, 131)


In [7]:
hardship_cols = [col for col in df.columns if col.startswith("hardship")]

df.drop(columns=hardship_cols, inplace=True, errors="ignore")

print("After dropping hardship columns:", df.shape)


After dropping hardship columns: (100000, 119)


## Create Target Variable

In [8]:
df["loan_default"] = df["loan_status"].apply(
    lambda x: 0 if x == "Fully Paid" else 1
)

df.drop(columns=["loan_status"], inplace=True)

print("Target created. Shape:", df.shape)


Target created. Shape: (100000, 119)


## Handle Missing Values

In [9]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include=["object"]).columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])


In [10]:
 # Drop columns with more than 70% missing values
missing_threshold = 0.7

missing_ratio = df.isna().mean()
high_missing_cols = missing_ratio[missing_ratio > missing_threshold].index.tolist()

print("Dropping columns:", high_missing_cols)

df.drop(columns=high_missing_cols, inplace=True)


Dropping columns: []


In [11]:
df.isna().sum().sort_values(ascending=False).head(10)


loan_amnt          0
funded_amnt        0
funded_amnt_inv    0
term               0
int_rate           0
installment        0
grade              0
sub_grade          0
emp_title          0
emp_length         0
dtype: int64

In [12]:
df.to_csv("loan_default_final_ready.csv", index=False)
print("Final corrected CSV saved")


Final corrected CSV saved
